## Genomics

Leukemia is a cancer of blood-generating tissues. Over 475,000 Americans have Leukemia or are in remission from it. It accounts for 3.3% of all new cancer cases and 3.8% of cancer deaths, with an estimated 66,890 new cases and 23,540 deaths in the U.S. in 2025.

There are two major leukemia families: Acute Lymphoblastic Leukemia (ALLB and ALLT, or ALL), which is cancer of immature lymphoid cells, and Acute Myeloid Leukemia (AML), which is cancer of cancer of immature myeloid cells.

Golub et al. (*Science*, 1999) popularized a dataset including about 7000 genes from 72 patients. The goal is to use genomics data to predict which patients are at risk of ALL versus AML, because the distinction is critical for timely and effective treatment.

1. Load the `golub.csv` dataset. Relabel all instances of ALLB and ALLT as 0, and all instances of ALL as 1. This is the target variable.

2. Use Linear Regression of the target variable on all of the genes provided. What is your mean squared error? Make a kernel density plot of your residuals, and a scatter plot comparing predicted and actual outcomes.

3. Use cross validation to compute the mean squared error of the linear model. Discuss your results from the perspective of the bias variance trade-off.

4. Use the cross validated LASSO to select a set of highly predictive genes. Which set of genes is selected? How many genes are discarded from the model? Make a scatterplot of your predictions versus the actual values.

5. Make a plot that shows the cross validated MSE as $alpha$ varies. For what values of $\alpha$ is the LASSO underfitting? Overfitting? What is the optimal penality hyperparameter that minimizes expected MSE?

6. Explain why linear regression performs perfectly on the training set, but the LASSO provides better predictions overall.

7. Why do regularization methods lend themselves to scenarios like precision health?

8. What are the risks of applying methods like the Lasso to precision health questions, where interventions will then be taken to optimize patient health?

In [ ]:
# 1
import pandas as pd
import numpy as np
import seaborn as sns
import seaborn as sns
import matplotlib.pyplot as plt


from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, LassoCV, lasso_path, Lasso, Ridge, RidgeCV
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('./undergrad_ml_assignments/data/golub.csv')
print(df.head())

mapper = {'allB':0,
          'allT':0,
          'aml':1}

df['outcome'] = df['cancer'].map(mapper)


# %

In [ ]:
#2

# 2. Straight linear regression

mse = lambda y,y_hat : np.mean( (y-y_hat)**2 )

y = df['outcome']
X = df.drop(['Samples', 'BM.PB', 'Gender', 'Source', 'tissue.mf', 'cancer','outcome'],axis=1)

model = LinearRegression()
reg = model.fit(X,y)

y_hat = reg.predict(X)
print(f'OLS training MSE: {mse(y,y_hat)}')

residuals = y_hat - y
sns.kdeplot(residuals)
plt.show()
residuals.describe()

sns.scatterplot(x=y,y=y_hat)
plt.show()

In [ ]:
# 3. Cross validation of the linear model

kfold = KFold(n_splits=5, shuffle=True, random_state=100) # Create folds
scores = cross_val_score( # Conduct kfcv:
    model,X,y, # Model and data
    cv=kfold, # Folds
    scoring='neg_mean_squared_error' # Loss function
)

mse = -scores

sns.histplot(mse,bins=10).set(title='MSE estiamtes')
plt.show()

print("Fold scores:", mse)
print("Mean score:", np.mean(mse))
print("Median score:", np.median(mse))
print("Std dev:", np.std(mse))


# %%

In [ ]:
# 4. Cross-Validated Lasso

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

alpha_grid = np.logspace(-4, -2, num=50)
model = LassoCV(cv=10,
                alphas=alpha_grid,
                random_state=100)
model = model.fit(X_sc, y)

alpha_star = model.alpha_
index_star = np.argmin( np.mean(model.mse_path_,axis=1) )
coefs_star = Lasso(alpha=alpha_star, max_iter=10000).fit(X_sc,y).coef_

# %%


In [ ]:
# 5. Cross validated MSE versus alpha

sns.lineplot( x=model.alphas_, y= np.mean(model.mse_path_,axis=1) )
plt.axvline(x=alpha_star, color='green', linestyle='--',
            linewidth=1.5)
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("Cross Validated MSE")
print(f'Optimal cost hyperparameter: {alpha_star}')
plt.show()

# %%

coefs = []
for alpha in model.alphas_: # For each alpha value,
    reg = Lasso(alpha=alpha, max_iter=10_000) # Create a lasso model
    reg = reg.fit(X_sc,y) # Run the regression
    coefs.append(reg.coef_) # Save the slope coefficients
coefs = np.array(coefs) # Cast list of lists to array

plt.figure()
for i in range(coefs.shape[1]):
    plt.plot(model.alphas_, coefs[:, i], label=X.columns[i]) # Switched in poly_names
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("Coefficient value")
plt.title("LASSO Coefficient Paths")
plt.axvline(x=alpha_star, color='green', linestyle='--')
plt.show()

# %%

coefs_star = coefs[index_star]
sns.histplot( coefs_star )
plt.show()

nonzero_indices = np.where( coefs_star != 0 )
print('Selected Genes:\n', X.columns[nonzero_indices])
sns.histplot( coefs_star[nonzero_indices] )
plt.show()

print("Nonzero coefficients:", np.sum(coefs_star != 0))
print("Total genes:", len(coefs_star))

for i in nonzero_indices:
    plt.plot(model.alphas_, coefs[:, i], label=X.columns[i])

plt.xscale("log")
plt.axvline(alpha_star, linestyle="--", color="green")
plt.xlabel("alpha")
plt.ylabel("Coefficient value")
plt.title("LASSO Paths (Selected Genes)")
plt.legend(fontsize=6)
plt.show()